# Phase 4 : table d'analyse et indicateurs

Ce notebook part des tables nettoyées de `data/clean/` (phase 3). Il construit la table d'analyse : une ligne par transaction, avec les informations du client et de la carte, la famille du commerçant, l'étiquette de fraude et les indicateurs de comportement. Les fonctions sont dans `src/features.py`. Chaque étape est expliquée dans `docs/`.

Les indicateurs qui regardent le passé (P4-04 à P4-07) n'utilisent que les transactions précédentes : une transaction n'entre jamais dans le calcul de son propre indicateur.

## Chargement des tables nettoyées

In [1]:
import time
import pandas as pd

from src.load_data import load_clean_table
from src.features import (
    build_analysis_table, check_analysis_table, add_time_features, check_time_features,
    add_age_at_transaction, check_age, add_amount_gap, check_amount_gap,
    add_card_frequency, add_recent_auth_errors, recount_card_activity,
    add_first_time_flags, check_first_time_flags, add_context_flags, check_context_flags,
    indicator_catalogue, describe_indicators, check_missing_and_bounds, save_analysis_table,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

In [2]:
df_transaction = load_clean_table("transactions")
df_users = load_clean_table("users")
df_card = load_clean_table("cards")
df_mcc = load_clean_table("mcc")
df_labels = load_clean_table("fraud_labels")
n_transactions = len(df_transaction)

for name, table in [("transactions", df_transaction), ("users", df_users), ("cards", df_card),
                    ("mcc", df_mcc), ("fraud_labels", df_labels)]:
    print(f"{name}: {table.shape[0]:,} lignes, {table.shape[1]} colonnes")

transactions: 13,305,915 lignes, 30 colonnes
users: 2,000 lignes, 14 colonnes
cards: 6,146 lignes, 11 colonnes
mcc: 109 lignes, 4 colonnes
fraud_labels: 8,914,963 lignes, 2 colonnes


## P4-01 : construire la table d'analyse

Les transactions reçoivent les colonnes utiles des clients, des cartes, la famille du code MCC et l'étiquette de fraude. Les colonnes brutes `merchant_state` et `zip` sont retirées : depuis P3-07, elles sont remplacées par `merchant_state_clean`, `merchant_country` et `zip_clean`.

Critère de fin : même nombre de lignes que la table des transactions. Voir `docs/P4-01_table_analyse.md`.

In [3]:
df = build_analysis_table(df_transaction, df_users, df_card, df_mcc, df_labels)
del df_transaction, df_labels

controle = check_analysis_table(df, n_transactions)
print("Critère de fin validé :", controle["statut"].all())
controle

Critère de fin validé : True


,contrôle,obtenu,statut
0,lignes de la table d'analyse,13305915,True
1,identifiants en double,0,True
2,valeurs vides dans les colonnes ajoutées,0,True
3,transactions étiquetées (is_fraud),8914963,True


In [4]:
print(f"Table d'analyse : {df.shape[0]:,} lignes, {df.shape[1]} colonnes, "
      f"{df.memory_usage(deep=True).sum() / 1e6:,.1f} Mo en mémoire")
df[["id", "date", "client_id", "card_id", "amount", "gender", "card_brand", "card_type",
    "mcc_family", "is_fraud"]].head()

Table d'analyse : 13,305,915 lignes, 44 colonnes, 1,334.3 Mo en mémoire


,id,date,client_id,card_id,amount,gender,card_brand,card_type,mcc_family,is_fraud
0,7475327,2010-01-01 00:01:00,1556,2972,-77.00,Female,Mastercard,Debit (Prepaid),Alimentation,0
1,7475328,2010-01-01 00:02:00,561,4575,14.57,Male,Mastercard,Credit,Grandes surfaces et commerces divers,0
2,7475329,2010-01-01 00:02:00,1129,102,80.00,Male,Mastercard,Debit,Services professionnels et financiers,0
3,7475331,2010-01-01 00:05:00,430,2860,200.00,Female,Mastercard,Debit,Services professionnels et financiers,<NA>
4,7475332,2010-01-01 00:06:00,848,3915,46.41,Male,Visa,Debit,Restauration et bars,0


**Constats**

- La table d'analyse a 13 305 915 lignes, comme la table des transactions : le critère de fin est validé.
- Aucun identifiant n'est en double et aucune colonne ajoutée n'est vide.
- 8 914 963 transactions ont une étiquette de fraude ; les autres ont une valeur vide dans `is_fraud`.
- La table a 44 colonnes et occupe 1 334,3 Mo en mémoire à cette étape.

## P4-02 : variables de temps

Année, mois, jour de la semaine (0 = lundi), heure, et deux indicateurs : `is_night` (de 0 h à 5 h 59) et `is_weekend` (samedi et dimanche).

Critère de fin : colonnes vérifiées sur un échantillon. Voir `docs/P4-02_variables_temps.md`.

In [5]:
df = add_time_features(df)

verif = check_time_features(df)
print("Critère de fin validé :", verif["identique au recalcul"].all())
verif

Critère de fin validé : True


,id,date,year,month,weekday,hour,is_night,is_weekend,identique au recalcul
12119134,22293358,2018-12-25 06:11:00,2018,12,1,6,0,0,True
11911763,22036862,2018-10-31 15:17:00,2018,10,2,15,0,0,True
11922349,22049988,2018-11-03 10:32:00,2018,11,5,10,0,1,True
5685601,14384899,2014-05-09 11:16:00,2014,5,4,11,0,0,True
706192,8320954,2010-07-30 10:15:00,2010,7,4,10,0,0,True
2047504,9947306,2011-08-18 10:17:00,2011,8,3,10,0,0,True
7068918,16081229,2015-05-12 16:51:00,2015,5,1,16,0,0,True
8054695,17289178,2016-01-25 22:31:00,2016,1,0,22,0,0,True
3933095,12245170,2013-01-22 20:16:00,2013,1,1,20,0,0,True
10050121,19741094,2017-07-02 06:58:00,2017,7,6,6,0,1,True


In [6]:
pd.Series({
    "transactions de nuit (0 h à 5 h 59)": int(df["is_night"].sum()),
    "transactions le week-end": int(df["is_weekend"].sum()),
    "transactions au total": len(df),
})

transactions de nuit (0 h à 5 h 59)      770383
transactions le week-end                3801414
transactions au total                  13305915
dtype: int64

**Constats**

- Les six variables de temps sont identiques au recalcul ligne par ligne sur les 10 transactions de l'échantillon.
- 770 383 transactions ont lieu la nuit (de 0 h à 5 h 59) et 3 801 414 le week-end, sur 13 305 915.